In [1]:
%pip install -U langchain-ollama
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_ollama import ChatOllama
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
import os
import certifi
from dotenv import load_dotenv

load_dotenv()

Note: you may need to restart the kernel to use updated packages.


True

In [2]:
from langchain.tools import tool
import requests

In [3]:
# SSL cert for API calls
os.environ["SSL_CERT_FILE"] = certifi.where()
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")


In [ ]:
# Tavily search tool
search_tool = TavilySearchResults()

# ollama LLM with free model
llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0,
)

In [6]:
REACT_PROMPT_TEMPLATE = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use this exact format:

Question: the input question you must answer
Thought: think about what to do
Action: one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought: {agent_scratchpad}"""

In [7]:
import requests
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get current weather for a city or country name."""
    
    geocoding_url = f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1&language=en&format=json"
    geocoding_response = requests.get(geocoding_url, timeout=10)
    geocoding_response.raise_for_status()
    geocoding_data = geocoding_response.json()

    if not geocoding_data.get("results"):
        return f"Error: Location '{location}' not found"

    result = geocoding_data["results"][0]
    city = result.get("name", location)
    country = result.get("country", "")
    latitude = result["latitude"]
    longitude = result["longitude"]
    city_name = f"{city}, {country}" if country else city

    weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m,relative_humidity_2m"
    weather_response = requests.get(weather_url, timeout=10)
    weather_response.raise_for_status()
    weather_data = weather_response.json()

    return f"Weather in {city_name}: Temp {weather_data['current']['temperature_2m']}°C, Wind {weather_data['current']['wind_speed_10m']} km/h, Humidity {weather_data['current']['relative_humidity_2m']}%"

In [8]:
tools = [search_tool,get_weather]

In [13]:
prompt = PromptTemplate.from_template(REACT_PROMPT_TEMPLATE)

agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=3,
    max_execution_time=30,
    handle_parsing_errors=True,
    early_stopping_method="generate"
)

result = agent_executor.invoke({"input": "what is the capital of india and what is the weather there now?"})




> Entering new AgentExecutor chain...
Thought: To answer this, I need to find out two pieces of information - the capital of India and the current weather in that city. I'll start with finding the capital.
Action: tavily_search_results_json
Action Input: "capital of india"[{'title': 'List of cities that served as the capital of India | India News - Times of India', 'url': 'https://timesofindia.indiatimes.com/india/list-of-cities-that-served-as-the-capital-of-india/articleshow/111815680.cms', 'content': "| Era | Capital(s) | Description |\n| Modern | Allahabad | Capital of India for a day in 1858, capital of North-Western Provinces. |\n| Calcutta | Capital of British India until 1911. |\n| Shimla | Summer capital by the latter half of the 19th century. |\n| New Delhi | Capital of India since 1911, buildings inaugurated in early 1931. |\n| Mediaeval | Agra | Capital of Sikandar Lodi during the Lodhi dynasty, moved from Delhi in 1506. |\n| Vijayanagara | Capital of Vijayanagara Empire u

In [14]:
print(result["output"])

The capital of India is New Delhi and the current weather in New Delhi, India is 35.9°C with a wind speed of 8.5 km/h and humidity at 40%.


In [15]:
import os
print(os.getenv("SSL_CERT_FILE"))

c:\Users\vgane\.conda\envs\langagent_new\lib\site-packages\certifi\cacert.pem
